# 02 · Data Cleaning

**Purpose:** Check the row order of the raw file, then save a cleaned, date-sorted copy to `data/processed/`. Every change is logged in `docs/cleaning_log.md`.

**Research questions served:** prepares the data for Q1, Q2 and Q3.

**Status:** complete.

## 1. Setup
We make the project files available, then import the libraries we need: pandas (tables), numpy (numbers), matplotlib (plots).

In [1]:
import os
# Colab: clone the repo. Codespaces: we are already inside it.
if not os.path.exists("data/raw"):
    if os.path.exists("../data/raw"):
        os.chdir("..")
    else:
        !git clone https://github.com/davis-mironga/wastewater-treatment-analytics.git
        %cd wastewater-treatment-analytics

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

RAW = Path("data/raw")
print(sorted(p.name for p in RAW.iterdir()))

['.gitkeep', 'CHECKSUMS.txt', 'water-treatment.data', 'water-treatment.names']


## 2. Is the file in date order?
The earliest date (1 Jan 1990) is earlier than the first row of the file (1 Mar 1990). We check whether the rows are shuffled, because time-based plots and control charts need the correct order.

In [2]:
cols = ["date",
 "Q-E","ZN-E","PH-E","DBO-E","DQO-E","SS-E","SSV-E","SED-E","COND-E",
 "PH-P","DBO-P","SS-P","SSV-P","SED-P","COND-P",
 "PH-D","DBO-D","DQO-D","SS-D","SSV-D","SED-D","COND-D",
 "PH-S","DBO-S","DQO-S","SS-S","SSV-S","SED-S","COND-S",
 "RD-DBO-P","RD-SS-P","RD-SED-P","RD-DBO-S","RD-DQO-S",
 "RD-DBO-G","RD-DQO-G","RD-SS-G","RD-SED-G"]

df = pd.read_csv(RAW / "water-treatment.data",
                 header=None, names=cols, na_values="?")
df["date_parsed"] = pd.to_datetime(
    df["date"].str.replace("D-", "", regex=False), format="%d/%m/%y")

print("In date order?", df["date_parsed"].is_monotonic_increasing)
back = df["date_parsed"].diff().dt.days < 0
print("Rows earlier than the previous row:", back.sum())
print(df.loc[back, ["date", "date_parsed"]].head())
print("First row:", df["date_parsed"].iloc[0].date(),
      "| Earliest:", df["date_parsed"].min().date())

In date order? False
Rows earlier than the previous row: 12
         date date_parsed
26   D-1/2/90  1990-02-01
49   D-1/1/90  1990-01-01
100  D-1/5/90  1990-05-01
126  D-1/4/90  1990-04-01
200  D-1/8/90  1990-08-01
First row: 1990-03-01 | Earliest: 1990-01-01


### What we see
The file is not in date order: 12 rows have a date earlier than the row above, and the file starts on 1 March 1990 although the earliest date is 1 January 1990. The next cell looks at these jumps in detail.

## 3. Where are the backward jumps, and what happens after sorting?
We list all 12 jumps, sort a copy by date, and check for duplicates and gaps.

In [3]:
# Keep the original file position so every row stays traceable
df["file_row"] = df.index

print("All 12 places where the date goes backwards:")
print(df.loc[back, ["file_row", "date", "date_parsed"]].to_string())

# Sort a copy by date
df_sorted = df.sort_values("date_parsed").reset_index(drop=True)
print()
print("Sorted copy in date order?", df_sorted["date_parsed"].is_monotonic_increasing)
print("Rows:", len(df_sorted), "| Duplicate dates:", df_sorted["date_parsed"].duplicated().sum())

# Where is the 32-day gap?
g = df_sorted["date_parsed"].diff().dt.days
for k in g[g > 3].index:
    print(f"Gap of {int(g[k])} days: {df_sorted['date_parsed'][k-1].date()} -> {df_sorted['date_parsed'][k].date()}")

All 12 places where the date goes backwards:
     file_row       date date_parsed
26         26   D-1/2/90  1990-02-01
49         49   D-1/1/90  1990-01-01
100       100   D-1/5/90  1990-05-01
126       126   D-1/4/90  1990-04-01
200       200   D-1/8/90  1990-08-01
249       249  D-1/11/90  1990-11-01
275       275  D-1/10/90  1990-10-01
325       325   D-1/2/91  1991-02-01
348       348   D-1/1/91  1991-01-01
402       402   D-1/4/91  1991-04-01
454       454   D-2/6/91  1991-06-02
502       502   D-1/8/91  1991-08-01

Sorted copy in date order? True
Rows: 527 | Duplicate dates: 0
Gap of 32 days: 1991-08-30 -> 1991-10-01


### What we see
Every backward jump falls at the start of a month, so the file consists of 13 ascending runs, each starting at a month boundary (some runs cover several months). The cause is unknown. A copy sorted by date has the same 527 rows, no duplicate dates, and one gap longer than 3 days: 30 Aug to 1 Oct 1991, so September 1991 has no records. All time-based analysis uses the sorted copy, with the original position kept in `file_row`. The raw file is unchanged.

## Cleaning summary
- The processed file `data/processed/water_treatment_clean.csv` is created by `src/cleaning.py`.
- It has 527 rows, sorted by date, with `?` read as missing and dates parsed.
- No values were imputed, removed or altered.
- Limitation: September 1991 has no records.